<a href="https://colab.research.google.com/github/biglalo104/Projects/blob/main/Hybrid%20KNN%E2%80%93LSTM%20Framework%20for%20Electricity%20Theft%20Detection%20in%20Smart%20Grids%20Using%20SGCC%20Smart%20Meter%20Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing  import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM,Dense,Dropout

In [3]:
#1.GENERATE MOCK SGCC DATA
#In the event that this was real data then here is where you would load the CSV
#Shape:(1000 users,365 days)
np.random.seed(42)
num_users = 1000
days = 365

#Create random consumption data
data = np.random.rand(num_users,days)*100

#Create Labels (0:Normal,1:Theft)-Imbalanced(5% theft)
labels = np.zeros(num_users)
labels[:50] = 1 # First 50 users are thieves

#Simulate theft behaviour (Lower consumption or erratic)
data[:50,:] = np.random.rand(50,days) *20

df = pd.DataFrame(data)
df['label'] = labels




In [5]:
#PREPROCESSING FOR LSTM
#LSTM requires 3D input:(Samples,Time Steps,Features)
X_lstm = df.drop('label',axis=1).values
y = df['label'].values

#Scale data between 0 and 1
scaler = MinMaxScaler()
X_lstm_scaled = scaler.fit_transform(X_lstm)

# Reshape for LSTM:(1000 samples,365 time_steps,1 feature)
X_lstm_reshaped = X_lstm_scaled.reshape((X_lstm_scaled.shape[0],X_lstm_scaled.shape[1],1))
X_train_lstm,X_test_lstm,y_train,y_test = train_test_split(X_lstm_reshaped,y,test_size=0.2,random_state=42)

In [6]:
#PREPROCESSING FOR KNN
#KNN works better on statistical features rather than raw 365-dimensional vectors
#We extract features like Mean,Std,Max,Min
X_knn_features = pd.DataFrame({
    'mean_usage':df.drop('label',axis=1).mean(axis=1),
    'std_usage':df.drop('label',axis=1).std(axis=1),
    'max_usage':df.drop('label',axis=1).max(axis=1),
    'min_usage':df.drop('label',axis=1).min(axis=1),
})

X_train_knn,X_test_knn,_,_=train_test_split(X_knn_features,y,test_size=0.2,random_state=42)

In [7]:
#4.BUILD LSTM MODEL
def build_lstm_model():
  model = Sequential([
      LSTM(64,activation='relu',input_shape=(days,1),return_sequences=True),
      Dropout(0.2),
      LSTM(32,activation='relu'),
      Dense(1,activation='sigmoid')#Sigmoid for binary Classification(Theft/No Theft)
  ])
  model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
  return model

lstm_model = build_lstm_model()
print("Training LSTM...")

#Note;In real implementation,use class_weight to handle imbalance
lstm_model.fit(X_train_lstm,y_train,epochs=10,batch_size=32,validation_split=0.1,verbose=1)

Training LSTM...
Epoch 1/10


/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


23/23 ━━━━━━━━━━━━━━━━━━━━ 7s 190ms/step - accuracy: 0.9444 - loss: 0.4528 - val_accuracy: 0.9500 - val_loss: nan
Epoch 2/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 199ms/step - accuracy: 0.9444 - loss: nan - val_accuracy: 0.9500 - val_loss: nan
Epoch 3/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 180ms/step - accuracy: 0.9444 - loss: nan - val_accuracy: 0.9500 - val_loss: nan
Epoch 4/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 190ms/step - accuracy: 0.9444 - loss: nan - val_accuracy: 0.9500 - val_loss: nan
Epoch 5/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 182ms/step - accuracy: 0.9444 - loss: nan - val_accuracy: 0.9500 - val_loss: nan
Epoch 6/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 176ms/step - accuracy: 0.9444 - loss: nan - val_accuracy: 0.9500 - val_loss: nan
Epoch 7/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 6s 200ms/step - accuracy: 0.9444 - loss: nan - val_accuracy: 0.9500 - val_loss: nan
Epoch 8/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 174ms/step - accuracy: 0.9444 - loss: nan - val_accuracy: 0.9500 - val_loss: nan
Epoch 9/10
23/23 ━━━━━━━━━━━━━━━

In [9]:
#BUILD KNN MODEL
print("Training KNN...")
#k=5 is standard
knn_model= KNeighborsClassifier(n_neighbors=5,weights='distance')
knn_model.fit(X_train_knn,y_train)


Training KNN...


KNeighborsClassifier(weights='distance')

In [10]:
#HYBRID ENSEMBLE PREDICTION
#Get probabilities from LSTM
lstm_probs = lstm_model.predict(X_test_lstm).flatten()

#Get probabilities from KNN
knn_probs =knn_model.predict_proba(X_test_knn)[:,1]

#Hybrid Strategy:Average the probabilities (Ensemble)
#You can weight them,e.g.,0.7*LSTM + 0.3*KNN
hybrid_probs = (lstm_probs + knn_probs)/2

#Convert probabilities to binary labels(Threshold 0.5)
hybrid_preds = (hybrid_probs>0.5).astype(int)
print("\n---Hybrid Model Results---")
print(classification_report(y_test,hybrid_preds))

7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 130ms/step

---Hybrid Model Results---
              precision    recall  f1-score   support

         0.0       0.97      1.00      0.98       194
         1.0       0.00      0.00      0.00         6

    accuracy                           0.97       200
   macro avg       0.48      0.50      0.49       200
weighted avg       0.94      0.97      0.96       200



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
